# Group 17: House Price Prediction (Ames Housing) — MLflow experiment tracking
FUSE AI-201 · Module 5/6 · **Cleaned and reproducible version**

**Data:** real Kaggle *House Prices - Advanced Regression Techniques* (Ames, Iowa) `train.csv`. No simulated data.
**How to run:** Runtime ▸ Restart and run all (Colab), or run top to bottom locally with `data/train.csv` present.
Run it **once, top to bottom** so each model is logged exactly once.

In [1]:
# Step 1 - install (only if missing) and import
import importlib.util, subprocess, sys
if importlib.util.find_spec("mlflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import sklearn
from mlflow.tracking import MlflowClient
from IPython.display import display

print("MLflow:", mlflow.__version__, "| scikit-learn:", sklearn.__version__)

MLflow: 3.17.2 | scikit-learn: 1.8.0


## Step 2 - Load the data
Works in Google Colab (upload dialog) or locally (`data/train.csv` or `train.csv`).

In [2]:
try:
    from google.colab import files          # only exists in Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_file(name):
    for path in (name, os.path.join("data", name)):
        if os.path.exists(path):
            return path
    return None

train_path = find_file("train.csv")
if train_path is None and IN_COLAB:
    uploaded = files.upload()
    assert "train.csv" in uploaded, "Please upload the Kaggle train.csv file."
    train_path = "train.csv"
assert train_path, "train.csv not found. Download it from Kaggle and place it in data/ or next to this notebook."

train_df = pd.read_csv(train_path)
print("Loaded:", train_path, "| shape:", train_df.shape)
assert "SalePrice" in train_df.columns
display(train_df.head())

Loaded: data/train.csv | shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Step 3 - Inspect the data

In [3]:
print("Rows and columns:", train_df.shape)
print("Duplicate rows:", train_df.duplicated().sum())
print("Missing target values:", train_df["SalePrice"].isna().sum())
missing = train_df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].head(20))

Rows and columns: (1460, 81)
Duplicate rows: 0
Missing target values: 0


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

## Step 4 - Features, target and ONE train/validation split
`Id` is an identifier, not a property characteristic, so it is dropped **everywhere** (the earlier notebook split the data twice and kept `Id` the second time).
Missing values and categories are handled inside the pipeline so the same steps apply to validation data and future houses.

In [4]:
from sklearn.model_selection import train_test_split

assert train_df["SalePrice"].notna().all()
X = train_df.drop(columns=["SalePrice", "Id"], errors="ignore")
y = train_df["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
print("Features:", X.shape[1], "| train houses:", len(X_train), "| validation houses:", len(X_valid))

Features: 79 | train houses: 1168 | validation houses: 292


## Step 5 - Preprocessing pipeline
1. Numeric: fill missing values with the median.
2. Categorical: fill missing with the most frequent value, then one-hot encode (unseen categories are ignored at prediction time).

In [5]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ])

def make_model(regressor):
    # log1p target transform: matches the log-based RMSLE metric used by Kaggle
    return TransformedTargetRegressor(
        regressor=Pipeline([("preprocessor", make_preprocessor()), ("model", regressor)]),
        func=np.log1p, inverse_func=np.expm1)

print("Numeric features:", len(numeric_features), "| categorical features:", len(categorical_features))

Numeric features: 36 | categorical features: 43


## Step 6 - Evaluation metrics
Primary metric: **RMSLE** (lower is better, used by Kaggle). RMSE and MAE are in dollars; R² is variance explained. All models are scored on the same validation set.

In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(model, X_valid, y_valid):
    predictions = np.maximum(model.predict(X_valid), 0)
    return predictions, {
        "RMSLE": float(np.sqrt(mean_squared_error(np.log1p(y_valid), np.log1p(predictions)))),
        "RMSE": float(np.sqrt(mean_squared_error(y_valid, predictions))),
        "MAE": float(mean_absolute_error(y_valid, predictions)),
        "R2": float(r2_score(y_valid, predictions)),
    }

## Step 7 - Configure MLflow (one experiment name)
A fresh experiment name is used so old FAILED / duplicate runs from earlier attempts do not appear in the comparison.

In [7]:
EXPERIMENT_NAME = "Group17_Ames_House_Prices_Final"
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri(), "| Experiment:", EXPERIMENT_NAME)

2026/10/30 11:50:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/30 11:50:51 INFO mlflow.store.db.utils: Updating database tables


2026/10/30 11:50:52 INFO mlflow.tracking.fluent: Experiment with name 'Group17_Ames_House_Prices_Final' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db | Experiment: Group17_Ames_House_Prices_Final


## Step 8 - Train and log both models (each exactly once)
Each run is inside a `with mlflow.start_run(...)` block, so it is closed as FINISHED (or FAILED only if training really fails).
Models are saved with `cloudpickle`; note that pickle-style files can execute code when loaded, so only load models you created yourself.

In [8]:
candidates = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}
# NOTE: if your original Random Forest used other settings, change them here and re-run.

results = {}
for run_name, regressor in candidates.items():
    model = make_model(regressor)
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        _, metrics = evaluate_model(model, X_valid, y_valid)
        mlflow.log_params({
            "model_type": type(regressor).__name__, "test_size": 0.2, "random_state": 42,
            "target_transform": "log1p", "n_train": len(X_train), "n_valid": len(X_valid),
            **{f"reg_{k}": v for k, v in regressor.get_params().items()
               if k in ("n_estimators", "max_depth", "min_samples_leaf")},
        })
        mlflow.log_metrics(metrics)
        info = mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")
        mlflow.set_tag("model_uri", info.model_uri)
    results[run_name] = metrics
    print(f"{run_name:17s}", {k: round(v, 4) for k, v in metrics.items()})

2026/10/30 11:50:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearRegression  {'RMSLE': 0.1419, 'RMSE': 25418.0437, 'MAE': 16969.9507, 'R2': 0.9158}


2026/10/30 11:51:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest      {'RMSLE': 0.1453, 'RMSE': 29705.1677, 'MAE': 17364.9403, 'R2': 0.885}


## Step 9 - Compare the MLflow runs
Sorted by lowest RMSLE. Do not assume the more complex model wins - use the measured validation results.
This comparison rests on **one** validation split; cross-validation and tuning would make it more reliable.

In [9]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME],
                          filter_string="attributes.status = 'FINISHED'",
                          order_by=["metrics.RMSLE ASC"])
cols = ["run_id", "tags.mlflow.runName", "status", "metrics.RMSLE", "metrics.RMSE", "metrics.MAE", "metrics.R2"]
display(runs[cols])
assert len(runs) == len(candidates), "Expected exactly one FINISHED run per model - restart and run once."

,run_id,tags.mlflow.runName,status,metrics.RMSLE,metrics.RMSE,metrics.MAE,metrics.R2
0,6576e21900884e9ea1bd0cab69901743,LinearRegression,FINISHED,0.141948,25418.043677,16969.950742,0.915769
1,fbe8117640c341e9b65376e38bdb11c5,RandomForest,FINISHED,0.145305,29705.167665,17364.940340,0.884960


## Step 10 - Load the best model back from MLflow and test it
This shows the saved experiment can be reused, not just the Python variable.

In [10]:
best_run = runs.iloc[0]
best_model_uri = best_run["tags.model_uri"]
best_model = mlflow.sklearn.load_model(best_model_uri)
print("Best model:", best_run["tags.mlflow.runName"], "| run:", best_run["run_id"])
print("Model URI:", best_model_uri)

sample = X_valid.head(5)
comparison = pd.DataFrame({
    "Actual": y_valid.head(5).values,
    "Predicted": np.round(np.maximum(best_model.predict(sample), 0)),
})
comparison["Error"] = comparison["Predicted"] - comparison["Actual"]
display(comparison)

Best model: LinearRegression | run: 6576e21900884e9ea1bd0cab69901743
Model URI: models:/m-49ada1e231fc471fb32edd968b67fb56


,Actual,Predicted,Error
0,154500,154182.0,-318.0
1,325000,333420.0,8420.0
2,115000,99767.0,-15233.0
3,159000,166101.0,7101.0
4,315500,340880.0,25380.0


## Step 11 - Optional: predict the Kaggle `test.csv`
Skipped automatically if `test.csv` is not available.

In [11]:
test_path = find_file("test.csv")
if test_path is None and IN_COLAB:
    print("Upload test.csv to create submission.csv (or skip this step).")
    up = files.upload()
    test_path = "test.csv" if "test.csv" in up else None

if test_path:
    test_df = pd.read_csv(test_path)
    X_kaggle_test = test_df.drop(columns=["Id"], errors="ignore").reindex(columns=X.columns)
    submission = pd.DataFrame({"Id": test_df["Id"],
                               "SalePrice": np.maximum(best_model.predict(X_kaggle_test), 0)})
    submission.to_csv("submission.csv", index=False)
    print("Submission shape:", submission.shape)
    display(submission.head())
else:
    print("test.csv not provided - skipping Kaggle submission step.")

test.csv not provided - skipping Kaggle submission step.


## Summary
- **Data:** real Kaggle Ames Housing `train.csv`, all 1,460 homes (no outlier removal), 79 features, no simulated data.
- **Models:** Linear Regression (baseline) and Random Forest (300 trees), both on a log1p target, one 80/20 split (1,168 / 292 homes).
- **Result (lowest validation RMSLE wins):** Linear Regression, RMSLE 0.1419, MAE $16,970, R² 0.916; Random Forest, RMSLE 0.1453, MAE $17,365, R² 0.885.
- **Role in the project:** a simple, fully tracked baseline. The submitted model is the tuned Lasso + LightGBM ensemble in the project root (cross-validated; held-out RMSLE 0.111). Figures differ because of the model, the split (outliers removed there) and tuning.
- **Limitations:** single split, no tuning, Ames 2006 to 2010 only. See the project README, section 9.
